# EGC Enzyme Tree

Reads `.a.ec.txt` files from `Metagenome_Samples`, builds the enzyme (EC number) tree using EC names and hierarchy from a local copy of KEGG's EC BRITE file (see `scripts/download_enzyme_brite.py`), and writes one `.enzyme.json` per sample.

**Run order:** Cell by cell top to bottom. Restart kernel before re-running.

In [ ]:
import json
import re
import time
from pathlib import Path
from collections import defaultdict
import sys
sys.path.append(r"C:\Users\sakoehn\Desktop\JGI\Notebooks")
from All_Coverages import allfold
from weighted_totals import calculate_denominator_egc
from enzyme_ec_utils import resolve_or_drop_ec

## 1. Load sample list

In [2]:
with open(r'C:\Users\sakoehn\Desktop\JGI\sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f"Loaded {len(samples)} samples")

Loaded 128 samples


## 2. Read enzyme annotation files → enz_db

`enz_db[gene_id]` holds three parallel lists — one entry per observation of that gene:
- `lookupId`: the EC label from column 2 of the `.a.ec.txt` file
- `timepoint`: the sample this observation came from
- `scaffolds`: the scaffold this gene sits on

Both branches of the if/else write to the same `enz_db` parameter so no observations are lost.

In [3]:
def read_enzyme(directpath, filename, s, enz_db):
    """
    Parse one sample's .a.ec.txt and append each gene observation to enz_db.
    Both branches write to the enz_db parameter — no shadowing.
    """
    path = directpath / filename
    if not path.exists():
        print(f"  WARNING: {path} not found, skipping")
        return

    with open(path, "r") as f:
        for line in f:
            parts   = line.rstrip("\n").split("\t")
            gene_id = parts[0]
            label   = parts[2]           # EC label, e.g. "EC:3.6.1.23"
            scaff   = gene_id.rsplit('_', 2)[0]

            if gene_id not in enz_db:
                enz_db[gene_id] = {
                    'lookupId':  [label],
                    'timepoint': [s],
                    'scaffolds': [scaff],
                }
            else:
                enz_db[gene_id]['lookupId'].append(label)
                enz_db[gene_id]['timepoint'].append(s)
                enz_db[gene_id]['scaffolds'].append(scaff)


base   = Path(r"C:\Users\sakoehn\Desktop\JGI\Metagenome_Samples")
enz_db = {}

for s in samples:
    read_enzyme(base / s, f"{s}.a.ec.txt", s, enz_db)

print(f"enz_db: {len(enz_db):,} genes")

enz_db: 22,293,001 genes


## 2.5 Load and parse EC BRITE

Reads the local copy of `ko01000.keg` (`DataStructures/Enzyme/ko01000.keg`, saved by `scripts/download_enzyme_brite.py` -- see that script for why this isn't a live fetch) and builds a flat `brite` dict with depth, category, parent_id, and parent_name — the same structure used by the KEGG tree. Every EC node has exactly one real parent in this file (unlike KO ids in KEGG's own hierarchy), so unlike EGC_KEGG_TREE.ipynb, no multi-parent handling is needed here.

In [4]:
def normalize_ec(ec):
    """Strip 'EC:' prefix if present, e.g. 'EC:3.6.1.23' → '3.6.1.23'."""
    return ec[3:] if ec.startswith("EC:") else ec


with open(r'C:\Users\sakoehn\Desktop\JGI\DataStructures\Enzyme\ko01000.keg', encoding='utf-8') as f:
    brite_file = f.read()

brite = {}
current_at_depth = {}

for raw_line in brite_file.splitlines():
    line = raw_line.strip()
    if not line or line[0] not in "ABCDEFGHIJK":
        continue

    level_char = line[0]
    depth = ord(level_char) - ord('A') + 1

    # Strip HTML tags before matching (A-level lines: "A<b>1. Oxidoreductases</b>")
    rest = re.sub(r"<[^>]+>", "", line[1:]).strip()
    if not rest:
        continue

    # W (2.6, docs/AUDIT-2026-09-22.md): the trailing name group used to be
    # required (`\s+(.+)$`), silently dropping any bare "<id>" line with no
    # name text -- ko01000.keg line 17308 ("C    7.2.3") is exactly that
    # (confirmed the only such line in the file). Skipping it left
    # current_at_depth[3] stale at the PREVIOUS named C-line, so the next
    # D-line ("7.2.3.1") inherited the WRONG parent. The name group is now
    # optional, falling back to the bare id as its own name -- same fix as
    # FastAPI/scripts/migrate_enzyme_paths_to_sql.py's iter_enzyme_path_rows.
    m = re.match(r"^(\d+(?:\.\d+){0,3})\.?(?:\s+(.+))?$", rest)
    if not m:
        continue

    node_id = m.group(1)
    name    = (m.group(2) or node_id).strip()

    parent      = current_at_depth.get(depth - 1)
    parent_id   = parent['id']   if parent else None
    parent_name = parent['name'] if parent else None
    category    = name if depth == 1 else current_at_depth.get(1, {}).get('name')

    current_at_depth[depth] = {'id': node_id, 'name': name}
    for d in list(current_at_depth.keys()):
        if d > depth:
            del current_at_depth[d]

    brite[node_id] = {
        'parent_id':   parent_id,
        'parent_name': parent_name,
        'name':        name,
        'category':    category,
        'depth':       depth,
    }

depth1 = {k: v for k, v in brite.items() if v['depth'] == 1}
print(f"Parsed {len(brite):,} BRITE entries  ({len(depth1)} depth-1 classes)")
for k, v in depth1.items():
    print(f"  {k}: {v['name']}")

Parsed 7,334 BRITE entries  (7 depth-1 classes)
  1: Oxidoreductases
  2: Transferases
  3: Hydrolases
  4: Lyases
  5: Isomerases
  6: Ligases
  7: Translocases


## 3. Build enzyme tree

For each gene in `enz_db`, strip the `EC:` prefix and look the bare EC number up in `brite`.  
Walk up the `parent_id` chain to the root, registering the gene and scaffold at **every ancestor node**.  
Only nodes with at least one observation from `enz_db` are added to the tree.

In [ ]:
enz_tree = {}  # {ec_id: {name, category, depth, parent_id, parent_name, timepoints}}
dropped_ec_ids   = set()  # ids absent from ko01000.keg's brite dict with no resolvable ancestor either
resolved_partial_ec_ids = set()  # E18/P1: ids successfully attributed to their deepest real ancestor (resolve_or_drop_ec, Notebooks/enzyme_ec_utils.py)

start = time.time()

for gene_id, ginfo in enz_db.items():
    lookups    = ginfo.get('lookupId',  [])
    timepoints = ginfo.get('timepoint', [])
    scaffolds  = ginfo.get('scaffolds', [])

    for ec_raw, sample, scaff in zip(lookups, timepoints, scaffolds):
        # E10 (docs/AUDIT.md): assert exactly one id per entry instead of
        # silently taking only the first of a ';'-joined list -- checked
        # against real data, this never currently fires (every entry has
        # exactly one id), so if it ever does, that's a real format change
        # worth failing loudly on rather than silently dropping ids.
        segments = ec_raw.split(';')
        assert len(segments) == 1, (
            f"Expected exactly one EC id per lookupId entry, got "
            f"{len(segments)} in {ec_raw!r} (gene {gene_id}) -- "
            f"the raw .a.ec.txt format may have changed."
        )
        ec_id = normalize_ec(segments[0].strip())

        # E18/P1 (docs/AUDIT-2026-10-05.md): resolve_or_drop_ec handles
        # all three cases in one real, tested implementation -- already a
        # real brite node, a partial/wildcard id (e.g. "1.1.1.-") or a
        # FULLY-specified id simply absent from this ko01000.keg download
        # (e.g. "3.6.4.12") -- instead of the notebook hand-rolling a
        # dash check that used to only cover the partial case. Checked
        # directly against sample 3300060358: 15 of 2,623 distinct
        # missing ids are fully-specified with a resolvable real ancestor
        # (e.g. "3.6.4.12"/".13" -> class "3.6.4"), undercounting node "3"
        # (Hydrolases) by 10.4% of its real gene count before this fix.
        start_id = resolve_or_drop_ec(ec_id, brite)
        if start_id is None:
            # Nothing resolves at any level -- the id truly isn't in this
            # ko01000.keg download at all. Still correctly dropped, just
            # made visible instead of silent (see docs/MISSING_ANNOTATIONS.md).
            dropped_ec_ids.add(ec_id)
            continue
        if start_id != ec_id:
            resolved_partial_ec_ids.add(ec_id)

        # Walk from the deepest REAL node (start_id -- the EC itself when
        # fully resolved, or its deepest resolved ancestor for a partial
        # id) up to the root, registering at every ancestor
        current = start_id
        while current is not None:
            if current not in brite:
                break

            if current not in enz_tree:
                b = brite[current]
                enz_tree[current] = {
                    'name':        b['name'],
                    'category':    b['category'],
                    'depth':       b['depth'],
                    'parent_id':   b['parent_id'],
                    'parent_name': b['parent_name'],
                    'timepoints':  {},
                }

            node = enz_tree[current]

            if sample not in node['timepoints']:
                node['timepoints'][sample] = {
                    'egc':       0,
                    'norm_egc':  0.0,
                    'scaffolds': {},
                    'genes':     set(),
                }

            tp = node['timepoints'][sample]
            # Only count this gene once per (EC node, sample) — a multi-domain
            # protein can produce multiple rows for the same EC in .a.ec.txt.
            if gene_id not in tp['genes']:
                tp['scaffolds'][scaff] = tp['scaffolds'].get(scaff, 0) + 1
            tp['genes'].add(gene_id)

            current = brite[current]['parent_id']

print(f"Enzyme tree: {len(enz_tree):,} nodes  ({time.time() - start:.1f}s)")
print(f"Resolved {len(resolved_partial_ec_ids)} distinct EC ids (partial or fully-specified) to their deepest real ancestor (E18/P1)")
print(f"Dropped {len(dropped_ec_ids)} distinct EC ids not found in ko01000.keg with no resolvable ancestor either (see docs/MISSING_ANNOTATIONS.md)")
if dropped_ec_ids:
    print(f"  e.g. {sorted(dropped_ec_ids)[:10]}")


## 4. Compute EGC and norm_egc

`egc = Σ (gene_count_on_scaffold × scaffold_coverage)`, using only genes annotated with an EC number (from this notebook's own `enz_db`).

`norm_egc = egc / weighted_totals[sample]`, where the denominator comes from `Notebooks/weighted_totals.py` (`calculate_denominator_egc`) and is the SAME value used by `EGC_KEGG_TREE.ipynb` / `EGC_COG_TREE.ipynb` / `EGC_TAX_TREE.ipynb` for a given sample -- computed once from every gene with any annotation across all four sources (via the mapping TSVs), not just genes with an EC hit. That shared scope is what makes norm_egc comparable across annotation types on the same plot. norm_egc still stays in (0, 1], since `egc` (an Enzyme-only subset of the shared total) can never exceed it.

In [6]:
# Cross-source-comparable denominator.
#
# calculate_denominator_egc() (Notebooks/weighted_totals.py) computes ONE
# shared total per sample from the mapping TSVs -- every gene annotated by
# ANY of the four sources (taxonomy/KEGG/enzyme/COG), counted once each --
# so this notebook uses the exact same denominator as EGC_KEGG_TREE.ipynb /
# EGC_COG_TREE.ipynb / EGC_TAX_TREE.ipynb for a given sample. That's what
# makes norm_egc comparable across annotation types on the same plot,
# instead of each notebook normalizing against only its own source's
# annotated genes.
weighted_totals = calculate_denominator_egc(samples)

# Sanity check on first sample: compare against the OLD unweighted,
# not-gene-count-weighted sum (allfold[sample]['total']) as a rough
# magnitude check -- not a correctness check on its own.
_s0 = next(iter(weighted_totals))
_new = weighted_totals[_s0]
_old = allfold.get(_s0, {}).get('total', 0)
print(f'Sample {_s0}: old denom={_old:,.0f}  new denom={_new:,.0f}')
print(f'norm_egc values will be ~{_old/_new:.2f}x smaller, now bounded by 1.0')

Sample 3300060358: old denom=8,298,642  new denom=16,137,978
norm_egc values will be ~0.51x smaller, now bounded by 1.0


In [7]:
def avg_gene_map(node_id):
    node = enz_tree[node_id]
    for sample_name, data in node['timepoints'].items():
        scaffold_counts = data.get('scaffolds')
        if not scaffold_counts or sample_name not in allfold:
            continue

        covs = allfold[sample_name]
        sample_total_depth = weighted_totals[sample_name]

        total_egc = 0
        for scaff, cnt in scaffold_counts.items():
            cov = covs.get(scaff, 0)
            total_egc += cnt * cov

        data['egc']      = total_egc
        data['norm_egc'] = (
            total_egc / sample_total_depth
            if sample_total_depth > 0 else 0.0
        )


for node_id in enz_tree:
    avg_gene_map(node_id)
print('EGC computation complete')


EGC computation complete


## 5. Write .enzyme.json files

In [8]:
base = Path(r"C:\Users\sakoehn\Desktop\JGI\DataStructures\Enzyme")
base.mkdir(parents=True, exist_ok=True)

for s in samples:
    tp_out = {}
    for ec_id, node in enz_tree.items():
        if s not in node['timepoints']:
            continue
        tp = node['timepoints'][s]
        tp_out[ec_id] = {
            'name':        node['name'],
            'depth':       node['depth'],
            'category':    node['category'],
            'parent_id':   node['parent_id'],
            'parent_name': node['parent_name'],
            'egc':         tp.get('egc',       0),
            'norm_egc':    tp.get('norm_egc',  0.0),
            'scaffolds':   tp.get('scaffolds', {}),
        }
    with open(base / f"{s}.enzyme.json", 'w') as f:
        json.dump(tp_out, f, indent=2)

print(f"Wrote {len(samples)} .enzyme.json files to {base}")

Wrote 128 .enzyme.json files to C:\Users\sakoehn\Desktop\JGI\DataStructures\Enzyme
